# cBioPortal Bulk RNA-seq in PyHealth

This notebook showcases a proposed PyHealth dataset + task contribution based on cBioPortal bulk RNA-seq data.

It demonstrates:

1. Dataset creation from downloaded cBioPortal study folders
2. Survival prediction task
3. Classification task
4. Simple MLP training with PyHealth
5. A small ablation over the number of selected genes

## Expected folder layout

```text
root/
  brca_tcga/
    data_mrna_seq_v2_rsem.txt
    data_clinical_patient.txt
  luad_tcga/
    data_mrna_seq_v2_rsem.txt
    data_clinical_patient.txt
```

This notebook assumes your custom files already exist in the PyHealth repo:

- `pyhealth/datasets/cbioportal_bulk_rna.py`
- `pyhealth/datasets/configs/cbioportal_bulk_rna.yaml`
- `pyhealth/tasks/bulk_rna_survival.py`
- `pyhealth/tasks/bulk_rna_classification.py`


In [30]:
# Basic imports
from pathlib import Path
import pprint

from pyhealth.datasets import get_dataloader
from pyhealth.datasets.splitter import split_by_patient
from pyhealth.models import MLP
from pyhealth.trainer import Trainer

from pyhealth.datasets.cbioportal_bulk_rna import CBioPortalBulkRNADataset
from pyhealth.tasks.bulk_rna_survival import BulkRNASurvivalPrediction
from pyhealth.tasks.bulk_rna_classification import BulkRNACancerClassification


## Configuration

Update these paths and study names to match your local data.


In [31]:
ROOT = "/Users/szymonszymura/Desktop/cancer_datasets/"  # change if needed
STUDY_DIRS = ["brca_tcga", "luad_tcga", "lusc_tcga"]
SEED = 42

ROOT_PATH = Path(ROOT)
assert ROOT_PATH.exists(), f"Root does not exist: {ROOT_PATH}"
print(ROOT_PATH)
print(STUDY_DIRS)


/Users/szymonszymura/Desktop/cancer_datasets
['brca_tcga', 'luad_tcga', 'lusc_tcga']


## 1. Create the dataset

The dataset implementation reads the raw cBioPortal files, finds common genes across cohorts, applies preprocessing, and writes a PyHealth-ready CSV.


In [32]:
dataset = CBioPortalBulkRNADataset(
    root=ROOT,
    study_dirs=STUDY_DIRS,
    top_k_genes=1000,
)

dataset.stats()


Found 17659 common genes across studies
Retained 1000 most variable genes
Saved 2087 processed samples to /Users/szymonszymura/Desktop/cancer_datasets/cbioportal_bulk_rna_samples-pyhealth.csv
Initializing cbioportal_bulk_rna dataset from /Users/szymonszymura/Desktop/cancer_datasets/ (dev mode: False)
No cache_dir provided. Using default cache dir: /Users/szymonszymura/Library/Caches/pyhealth/573459d5-6ad7-595a-ac24-680dc30c38e1
Found cached event dataframe: /Users/szymonszymura/Library/Caches/pyhealth/573459d5-6ad7-595a-ac24-680dc30c38e1/global_event_df.parquet
Dataset: cbioportal_bulk_rna
Dev mode: False
Number of patients: 1603
Number of events: 1603


Inspect one patient record after BaseDataset loading.


In [33]:
patient_id = dataset.unique_patient_ids[0]
patient = dataset.get_patient(patient_id)
print("Patient ID:", patient_id)
patient.data_source.head()


Found 1603 unique patient IDs
Patient ID: TCGA-EW-A1IW


patient_id,event_type,timestamp,samples/sample_id,samples/cancer_type,samples/subtype,samples/os_months,samples/dfs_months,samples/os_status,samples/dfs_status,samples/expression_json
str,str,datetime[ns],str,str,str,str,str,str,str,str
"""TCGA-EW-A1IW""","""samples""",null,"""TCGA-EW-A1IW-01""","""brca_tcga""",null,"""12.19""","""12.19""","""0:LIVING""","""0:DiseaseFree""","""[-0.4743327819687335, -0.75955…"


## 2. Survival prediction task

The current survival task is a discretized classification task built from overall survival months and status.


In [34]:
survival_task = BulkRNASurvivalPrediction()
survival_samples = survival_task(patient)
pprint.pp(survival_samples[:1])


[{'patient_id': 'TCGA-EW-A1IW',
  'x': [-0.4743327819687335,
        -0.7595562611654536,
        1.2840519909477888,
        -0.6822908386079807,
        1.237372212086866,
        1.3388780674072722,
        1.779045276200557,
        -0.46328492068236293,
        0.0024144647251930483,
        -0.33440663051011976,
        -0.1527425936761222,
        -1.1413444826788235,
        0.349023218095966,
        -0.5352065965110233,
        -0.7880151428932135,
        0.5914679751071555,
        1.1580069633035528,
        -0.5135888316136572,
        0.2547957092842164,
        -0.45097687019594346,
        0.22064039239927932,
        1.3798479185552457,
        0.21088299243866482,
        1.239575309957627,
        0.19225141132867868,
        1.662596744742977,
        1.115044704374988,
        0.35457736134111745,
        1.4771479065978588,
        -0.20103459577554145,
        -0.9938654790749795,
        1.101730712817522,
        0.47534842011612416,
        -1.000645061373952

Turn the dataset into a PyHealth `SampleDataset`.


In [35]:
survival_sample_dataset = dataset.set_task(survival_task)
print("Number of survival samples:", len(survival_sample_dataset))
pprint.pp(survival_sample_dataset[0])


Setting task bulk_rna_survival_prediction for cbioportal_bulk_rna base dataset...
Task cache paths: task_df=/Users/szymonszymura/Library/Caches/pyhealth/573459d5-6ad7-595a-ac24-680dc30c38e1/tasks/bulk_rna_survival_prediction_3a991655-f909-5c9f-ad25-88ed4ecf7151/task_df.ld, samples=/Users/szymonszymura/Library/Caches/pyhealth/573459d5-6ad7-595a-ac24-680dc30c38e1/tasks/bulk_rna_survival_prediction_3a991655-f909-5c9f-ad25-88ed4ecf7151/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Found cached processed samples at /Users/szymonszymura/Library/Caches/pyhealth/573459d5-6ad7-595a-ac24-680dc30c38e1/tasks/bulk_rna_survival_prediction_3a991655-f909-5c9f-ad25-88ed4ecf7151/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld, skipping processing.
Number of survival samples: 1593
{'patient_id': 'TCGA-05-4244',
 'x': tensor([-7.8467e-01,  2.9434e-03,  7.9970e-01,  1.5858e+00,  5.1495e-01,
        -1.2353e+00,  1.6196e-01, -4.7567e-01, -5.3215e-01,  1.8887e+00,
        -1.4016e+00,  1.5676e+00,  2.8490e

Split into train/validation/test and create dataloaders.


In [36]:
train_ds, val_ds, test_ds = split_by_patient(
    survival_sample_dataset,
    [0.7, 0.1, 0.2],
    seed=SEED,
)

train_loader = get_dataloader(train_ds, batch_size=32, shuffle=True)
val_loader = get_dataloader(val_ds, batch_size=32, shuffle=False)
test_loader = get_dataloader(test_ds, batch_size=32, shuffle=False)

print(len(train_ds), len(val_ds), len(test_ds))


1115 159 319


Train a simple MLP baseline.


In [37]:
survival_model = MLP(
    dataset=survival_sample_dataset,
    feature_keys=["x"],
    label_key="y",
    mode="multiclass",
    hidden_dim=128,
)

survival_trainer = Trainer(
    model=survival_model,
    metrics=["accuracy", "f1_macro"],
    device="cpu",
)

survival_trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    test_dataloader=test_loader,
    epochs=5,
    monitor="f1_macro",
    monitor_criterion="max",
)

survival_results = survival_trainer.evaluate(test_loader)
survival_results


MLP(
  (embedding_model): EmbeddingModel(embedding_layers=ModuleDict(
    (x): Linear(in_features=1000, out_features=128, bias=True)
  ))
  (activation): ReLU()
  (mlp): ModuleDict(
    (x): Sequential(
      (0): Linear(in_features=128, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=128, bias=True)
    )
  )
  (fc): Linear(in_features=128, out_features=3, bias=True)
)
Metrics: ['accuracy', 'f1_macro']
Device: cpu

Training:
Batch size: 32
Optimizer: <class 'torch.optim.adam.Adam'>
Optimizer params: {'lr': 0.001}
Weight decay: 0.0
Max grad norm: None
Val dataloader: <torch.utils.data.dataloader.DataLoader object at 0x391eed850>
Monitor: f1_macro
Monitor criterion: max
Epochs: 5
Patience: None



Epoch 0 / 5:   0%|          | 0/35 [00:00<?, ?it/s]

--- Train epoch-0, step-35 ---
loss: 1.0483


Evaluation: 100%|████████████████████████████████| 5/5 [00:00<00:00, 526.86it/s]

--- Eval epoch-0, step-35 ---
accuracy: 0.4969
f1_macro: 0.3474
loss: 1.0559
New best f1_macro score (0.3474) at epoch-0, step-35



...

--- Eval epoch-4, step-175 ---
accuracy: 0.4214
f1_macro: 0.3657
loss: 1.3797
Loaded best model



Evaluation: 100%|██████████████████████████████| 10/10 [00:00<00:00, 555.38it/s]

--- Test ---
accuracy: 0.4608
f1_macro: 0.4010
loss: 1.1308



Evaluation: 100%|██████████████████████████████| 10/10 [00:00<00:00, 536.40it/s]


{'accuracy': 0.4608150470219436,
 'f1_macro': 0.4010321448069553,
 'loss': 1.1308193862438203}

## 3. Classification task

You can classify either:

- `cancer_type` across cohorts
- `subtype` within cohorts where subtype labels exist


In [38]:
classification_task = BulkRNACancerClassification(label_field="cancer_type")
classification_samples = classification_task(patient)
pprint.pp(classification_samples[:1])


[{'patient_id': 'TCGA-EW-A1IW',
  'x': [-0.4743327819687335,
        -0.7595562611654536,
        1.2840519909477888,
        -0.6822908386079807,
        1.237372212086866,
        1.3388780674072722,
        1.779045276200557,
        -0.46328492068236293,
        0.0024144647251930483,
        -0.33440663051011976,
        -0.1527425936761222,
        -1.1413444826788235,
        0.349023218095966,
        -0.5352065965110233,
        -0.7880151428932135,
        0.5914679751071555,
        1.1580069633035528,
        -0.5135888316136572,
        0.2547957092842164,
        -0.45097687019594346,
        0.22064039239927932,
        1.3798479185552457,
        0.21088299243866482,
        1.239575309957627,
        0.19225141132867868,
        1.662596744742977,
        1.115044704374988,
        0.35457736134111745,
        1.4771479065978588,
        -0.20103459577554145,
        -0.9938654790749795,
        1.101730712817522,
        0.47534842011612416,
        -1.000645061373952

In [39]:
classification_sample_dataset = dataset.set_task(classification_task)
print("Number of classification samples:", len(classification_sample_dataset))
pprint.pp(classification_sample_dataset[0])


Setting task bulk_rna_cancer_classification for cbioportal_bulk_rna base dataset...
Task cache paths: task_df=/Users/szymonszymura/Library/Caches/pyhealth/573459d5-6ad7-595a-ac24-680dc30c38e1/tasks/bulk_rna_cancer_classification_d54be90c-338b-580d-a407-5ece9ac861db/task_df.ld, samples=/Users/szymonszymura/Library/Caches/pyhealth/573459d5-6ad7-595a-ac24-680dc30c38e1/tasks/bulk_rna_cancer_classification_d54be90c-338b-580d-a407-5ece9ac861db/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Found cached processed samples at /Users/szymonszymura/Library/Caches/pyhealth/573459d5-6ad7-595a-ac24-680dc30c38e1/tasks/bulk_rna_cancer_classification_d54be90c-338b-580d-a407-5ece9ac861db/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld, skipping processing.
Number of classification samples: 1603
{'patient_id': 'TCGA-05-4244',
 'x': tensor([-7.8467e-01,  2.9434e-03,  7.9970e-01,  1.5858e+00,  5.1495e-01,
        -1.2353e+00,  1.6196e-01, -4.7567e-01, -5.3215e-01,  1.8887e+00,
        -1.4016e+00,  1.5676

Train the classification baseline.


In [40]:
train_ds_c, val_ds_c, test_ds_c = split_by_patient(
    classification_sample_dataset,
    [0.7, 0.1, 0.2],
    seed=SEED,
)

train_loader_c = get_dataloader(train_ds_c, batch_size=32, shuffle=True)
val_loader_c = get_dataloader(val_ds_c, batch_size=32, shuffle=False)
test_loader_c = get_dataloader(test_ds_c, batch_size=32, shuffle=False)

classification_model = MLP(
    dataset=classification_sample_dataset,
    feature_keys=["x"],
    label_key="y",
    mode="multiclass",
    hidden_dim=128,
)

classification_trainer = Trainer(
    model=classification_model,
    metrics=["accuracy", "f1_macro"],
    device="cpu",
)

classification_trainer.train(
    train_dataloader=train_loader_c,
    val_dataloader=val_loader_c,
    test_dataloader=test_loader_c,
    epochs=5,
    monitor="f1_macro",
    monitor_criterion="max",
)

classification_results = classification_trainer.evaluate(test_loader_c)
classification_results


MLP(
  (embedding_model): EmbeddingModel(embedding_layers=ModuleDict(
    (x): Linear(in_features=1000, out_features=128, bias=True)
  ))
  (activation): ReLU()
  (mlp): ModuleDict(
    (x): Sequential(
      (0): Linear(in_features=128, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=128, bias=True)
    )
  )
  (fc): Linear(in_features=128, out_features=2, bias=True)
)
Metrics: ['accuracy', 'f1_macro']
Device: cpu

Training:
Batch size: 32
Optimizer: <class 'torch.optim.adam.Adam'>
Optimizer params: {'lr': 0.001}
Weight decay: 0.0
Max grad norm: None
Val dataloader: <torch.utils.data.dataloader.DataLoader object at 0x38cda4650>
Monitor: f1_macro
Monitor criterion: max
Epochs: 5
Patience: None



Epoch 0 / 5:   0%|          | 0/36 [00:00<?, ?it/s]

--- Train epoch-0, step-36 ---
loss: 0.0472


Evaluation: 100%|████████████████████████████████| 5/5 [00:00<00:00, 548.79it/s]

--- Eval epoch-0, step-36 ---
accuracy: 1.0000
f1_macro: 1.0000
loss: 0.0000
New best f1_macro score (1.0000) at epoch-0, step-36



...

--- Eval epoch-4, step-180 ---
accuracy: 0.9938
f1_macro: 0.9928
loss: 0.0064
Loaded best model



Evaluation: 100%|██████████████████████████████| 11/11 [00:00<00:00, 563.85it/s]

--- Test ---
accuracy: 1.0000
f1_macro: 1.0000
loss: 0.0000



Evaluation: 100%|██████████████████████████████| 11/11 [00:00<00:00, 593.70it/s]


{'accuracy': 1.0, 'f1_macro': 1.0, 'loss': 5.655619824987665e-08}

## 4. Small ablation study

Vary the number of selected genes for downstream tasks.


In [41]:
TOP_K_LIST = [500, 1000, 5000]
ablation_results = []

for top_k in TOP_K_LIST:
    print(f"\n=== top_k_genes={top_k} ===")
    cache_dir = Path(".cache") / f"notebook_survival_topk_{top_k}"

    ds = CBioPortalBulkRNADataset(
        root=ROOT,
        study_dirs=STUDY_DIRS,
        top_k_genes=top_k,
        cache_dir=cache_dir,
    )

    task = BulkRNASurvivalPrediction()
    sample_ds = ds.set_task(task)

    tr, va, te = split_by_patient(sample_ds, [0.7, 0.1, 0.2], seed=SEED)
    tr_loader = get_dataloader(tr, batch_size=32, shuffle=True)
    va_loader = get_dataloader(va, batch_size=32, shuffle=False)
    te_loader = get_dataloader(te, batch_size=32, shuffle=False)

    model = MLP(
        dataset=sample_ds,
        feature_keys=["x"],
        label_key="y",
        mode="multiclass",
        hidden_dim=128,
    )

    trainer = Trainer(
        model=model,
        metrics=["accuracy", "f1_macro"],
        device="cpu",
    )

    trainer.train(
        train_dataloader=tr_loader,
        val_dataloader=va_loader,
        test_dataloader=te_loader,
        epochs=5,
        monitor="f1_macro",
        monitor_criterion="max",
    )

    result = trainer.evaluate(te_loader)
    result["top_k_genes"] = top_k
    ablation_results.append(result)

ablation_results



=== top_k_genes=500 ===
Initializing cbioportal_bulk_rna dataset from /Users/szymonszymura/Desktop/cancer_datasets/ (dev mode: False)
Using provided cache_dir: .cache/notebook_survival_topk_500/573459d5-6ad7-595a-ac24-680dc30c38e1
Setting task bulk_rna_survival_prediction for cbioportal_bulk_rna base dataset...
Task cache paths: task_df=.cache/notebook_survival_topk_500/573459d5-6ad7-595a-ac24-680dc30c38e1/tasks/bulk_rna_survival_prediction_3a991655-f909-5c9f-ad25-88ed4ecf7151/task_df.ld, samples=.cache/notebook_survival_topk_500/573459d5-6ad7-595a-ac24-680dc30c38e1/tasks/bulk_rna_survival_prediction_3a991655-f909-5c9f-ad25-88ed4ecf7151/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Found cached processed samples at .cache/notebook_survival_topk_500/573459d5-6ad7-595a-ac24-680dc30c38e1/tasks/bulk_rna_survival_prediction_3a991655-f909-5c9f-ad25-88ed4ecf7151/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld, skipping processing.
MLP(
  (embedding_model): EmbeddingModel(embedding_layers=M

Epoch 0 / 5:   0%|          | 0/35 [00:00<?, ?it/s]

--- Train epoch-0, step-35 ---
loss: 1.0452


Evaluation: 100%|████████████████████████████████| 5/5 [00:00<00:00, 497.71it/s]

--- Eval epoch-0, step-35 ---
accuracy: 0.4654
f1_macro: 0.2748
loss: 1.0308
New best f1_macro score (0.2748) at epoch-0, step-35



...

--- Eval epoch-4, step-175 ---
accuracy: 0.4214
f1_macro: 0.3774
loss: 1.7151
New best f1_macro score (0.3774) at epoch-4, step-175
Loaded best model



Evaluation: 100%|██████████████████████████████| 10/10 [00:00<00:00, 563.67it/s]

--- Test ---
accuracy: 0.4075
f1_macro: 0.3647
loss: 1.8377



Evaluation: 100%|██████████████████████████████| 10/10 [00:00<00:00, 565.23it/s]


[{'accuracy': 0.4200626959247649,
  'f1_macro': 0.37308185100464114,
  'loss': 1.4741294503211975,
  'top_k_genes': 500},
 {'accuracy': 0.438871473354232,
  'f1_macro': 0.3933398926328375,
  'loss': 1.8205282092094421,
  'top_k_genes': 1000},
 {'accuracy': 0.40752351097178685,
  'f1_macro': 0.3647008821324797,
  'loss': 1.8377392530441283,
  'top_k_genes': 5000}]

Result: For three cancer types tested and default windows for survival bins, varying number of target genes used as features for modeling results in slight differences in model performance with best performance using 1000 common genes, and lower performance from 500 common genes (likely due to too small number of features) and 5000 common genes (likely due to too much noise from too many features). Overall survival prediction in this setting has quite low accuracy.

Vary number of genes and bin edges for survival analysis

In [42]:
TOP_K_LIST = [500, 1000, 5000]
ablation_results = []

for top_k in TOP_K_LIST:
    print(f"\n=== top_k_genes={top_k} ===")
    cache_dir = Path(".cache") / f"notebook_survival_topk_{top_k}"

    ds = CBioPortalBulkRNADataset(
        root=ROOT,
        study_dirs=STUDY_DIRS,
        top_k_genes=top_k,
        cache_dir=cache_dir,
    )

    task = BulkRNASurvivalPrediction(bin_edges=[24.0, 48.0])
    sample_ds = ds.set_task(task)

    tr, va, te = split_by_patient(sample_ds, [0.7, 0.1, 0.2], seed=SEED)
    tr_loader = get_dataloader(tr, batch_size=32, shuffle=True)
    va_loader = get_dataloader(va, batch_size=32, shuffle=False)
    te_loader = get_dataloader(te, batch_size=32, shuffle=False)

    model = MLP(
        dataset=sample_ds,
        feature_keys=["x"],
        label_key="y",
        mode="multiclass",
        hidden_dim=128,
    )

    trainer = Trainer(
        model=model,
        metrics=["accuracy", "f1_macro"],
        device="cpu",
    )

    trainer.train(
        train_dataloader=tr_loader,
        val_dataloader=va_loader,
        test_dataloader=te_loader,
        epochs=5,
        monitor="f1_macro",
        monitor_criterion="max",
    )

    result = trainer.evaluate(te_loader)
    result["top_k_genes"] = top_k
    ablation_results.append(result)

ablation_results



=== top_k_genes=500 ===
Initializing cbioportal_bulk_rna dataset from /Users/szymonszymura/Desktop/cancer_datasets/ (dev mode: False)
Using provided cache_dir: .cache/notebook_survival_topk_500/573459d5-6ad7-595a-ac24-680dc30c38e1
Setting task bulk_rna_survival_prediction for cbioportal_bulk_rna base dataset...
Task cache paths: task_df=.cache/notebook_survival_topk_500/573459d5-6ad7-595a-ac24-680dc30c38e1/tasks/bulk_rna_survival_prediction_a3a4de18-2860-589a-bbef-7fbbb6a57361/task_df.ld, samples=.cache/notebook_survival_topk_500/573459d5-6ad7-595a-ac24-680dc30c38e1/tasks/bulk_rna_survival_prediction_a3a4de18-2860-589a-bbef-7fbbb6a57361/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Applying task transformations on data with 1 workers...
Found cached event dataframe: .cache/notebook_survival_topk_500/573459d5-6ad7-595a-ac24-680dc30c38e1/global_event_df.parquet
Detected Jupyter notebook environment, setting num_workers to 1
Single worker mode, processing sequentially
Worker 0 started 

  0%|                                                  | 0/1603 [00:00<?, ?it/s]

Rank 0 inferred the following `['bytes']` data format.


100%|██████████████████████████████████████| 1603/1603 [00:01<00:00, 864.08it/s]

Worker 0 finished processing patients.
Fitting processors on the dataset...


...

--- Eval epoch-4, step-175 ---
accuracy: 0.4277
f1_macro: 0.3821
loss: 1.6269
Loaded best model



Evaluation: 100%|██████████████████████████████| 10/10 [00:00<00:00, 550.72it/s]

--- Test ---
accuracy: 0.5016
f1_macro: 0.4134
loss: 1.0738



Evaluation: 100%|██████████████████████████████| 10/10 [00:00<00:00, 551.55it/s]


[{'accuracy': 0.4200626959247649,
  'f1_macro': 0.39129409013658706,
  'loss': 1.8806544065475463,
  'top_k_genes': 500},
 {'accuracy': 0.4169278996865204,
  'f1_macro': 0.3895674336643166,
  'loss': 1.9002004742622376,
  'top_k_genes': 1000},
 {'accuracy': 0.5015673981191222,
  'f1_macro': 0.413445350945351,
  'loss': 1.0738124787807464,
  'top_k_genes': 5000}]

Result: Changing bin edges for survival analysis appears to improve model performance for 5000 top shared genes as features. Those results need to be interpreted with caution as the model might learn to distinguish between cancers (if their survival is quite different) 

Vary number of genes for classification task

In [43]:
TOP_K_LIST = [500, 1000, 5000]
ablation_results = []

for top_k in TOP_K_LIST:
    print(f"\n=== top_k_genes={top_k} ===")

    cache_dir = Path(".cache") / f"classification_topk_{top_k}"

    ds = CBioPortalBulkRNADataset(
        root=ROOT,
        study_dirs=STUDY_DIRS,
        top_k_genes=top_k,
        cache_dir=cache_dir,
    )

    task = BulkRNACancerClassification(label_field="cancer_type")
    sample_ds = ds.set_task(task)

    tr, va, te = split_by_patient(sample_ds, [0.7, 0.1, 0.2], seed=SEED)
    tr_loader = get_dataloader(tr, batch_size=32, shuffle=True)
    va_loader = get_dataloader(va, batch_size=32, shuffle=False)
    te_loader = get_dataloader(te, batch_size=32, shuffle=False)

    model = MLP(
        dataset=sample_ds,
        feature_keys=["x"],
        label_key="y",
        mode="multiclass",
        hidden_dim=128,
    )

    trainer = Trainer(
        model=model,
        metrics=["accuracy", "f1_macro"],
        device="cpu",
    )

    trainer.train(
        train_dataloader=tr_loader,
        val_dataloader=va_loader,
        test_dataloader=te_loader,
        epochs=5,
        monitor="f1_macro",
        monitor_criterion="max",
    )

    result = trainer.evaluate(te_loader)
    result["top_k_genes"] = top_k
    ablation_results.append(result)

ablation_results


=== top_k_genes=500 ===
Initializing cbioportal_bulk_rna dataset from /Users/szymonszymura/Desktop/cancer_datasets/ (dev mode: False)
Using provided cache_dir: .cache/classification_topk_500/573459d5-6ad7-595a-ac24-680dc30c38e1
Setting task bulk_rna_cancer_classification for cbioportal_bulk_rna base dataset...
Task cache paths: task_df=.cache/classification_topk_500/573459d5-6ad7-595a-ac24-680dc30c38e1/tasks/bulk_rna_cancer_classification_d54be90c-338b-580d-a407-5ece9ac861db/task_df.ld, samples=.cache/classification_topk_500/573459d5-6ad7-595a-ac24-680dc30c38e1/tasks/bulk_rna_cancer_classification_d54be90c-338b-580d-a407-5ece9ac861db/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Applying task transformations on data with 1 workers...
No cached event dataframe found. Creating: .cache/classification_topk_500/573459d5-6ad7-595a-ac24-680dc30c38e1/global_event_df.parquet
Scanning table: samples from /Users/szymonszymura/Desktop/cancer_datasets/cbioportal_bulk_rna_samples-pyhealth.csv
Cac

  0%|                                                  | 0/2087 [00:00<?, ?it/s]

Rank 0 inferred the following `['bytes']` data format.


100%|██████████████████████████████████████| 2087/2087 [00:02<00:00, 922.66it/s]

Worker 0 finished processing patients.
Fitting processors on the dataset...


...

--- Eval epoch-4, step-230 ---
accuracy: 0.9665
f1_macro: 0.9553
loss: 0.1269
Loaded best model



Evaluation: 100%|██████████████████████████████| 14/14 [00:00<00:00, 642.31it/s]

--- Test ---
accuracy: 0.9641
f1_macro: 0.9500
loss: 0.0787



Evaluation: 100%|██████████████████████████████| 14/14 [00:00<00:00, 625.68it/s]


[{'accuracy': 0.9641148325358851,
  'f1_macro': 0.9500033594455473,
  'loss': 0.07670222285800524,
  'top_k_genes': 500},
 {'accuracy': 0.9617224880382775,
  'f1_macro': 0.9458530887102315,
  'loss': 0.16668161856276623,
  'top_k_genes': 1000},
 {'accuracy': 0.9641148325358851,
  'f1_macro': 0.950034615614395,
  'loss': 0.07872871242817123,
  'top_k_genes': 5000}]

Results: In the context of cancer classification model appears to have comparable high performance for each of the number of common genes tested. These results need to be interpreted with caution, as classification between different cancer types might be trivial (for instance tissue specific genes might drive classification, for example breast cancer vs lung cancer might be classified based on breast tissue vs lung tissue specific genes alone. More sophisticated classifier would allow to distinguish between subtypes of given cancer).